# Freight Rate Extraction Using OpenAI Structured Outputs

This notebook demonstrates **deterministic extraction** of FCL freight rates from tariff PDFs using:
- OpenAI's Structured Outputs API (gpt-4o-2024-08-06)
- Pydantic models for type-safe DTOs
- Regional date awareness for accurate effective periods

## 1. Setup and Dependencies

In [ ]:
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, Field
from typing import List, Optional, Literal
from datetime import date
from enum import Enum
import json
import base64

load_dotenv(override=True)

## 2. Define Data Models (VendorRate DTO)

These Pydantic models define the **strict schema** for structured outputs.

In [ ]:
class ChargeType(str, Enum):
    OCEAN_FREIGHT = "ocean_freight"
    DG3_SURCHARGE = "dg3_surcharge"
    DG2_SURCHARGE = "dg2_surcharge"
    BAF_LSS = "baf_lss"
    AMS = "ams"

class ContainerType(str, Enum):
    GP20 = "20'GP"
    GP40 = "40'GP"
    HC40 = "40'HC"

class Currency(str, Enum):
    USD = "USD"
    SGD = "SGD"

class Region(str, Enum):
    SEA_CHINA = "sea_china"
    INDIA_MIDDLE_EAST_SUBCON = "india_middle_east_subcon"

In [ ]:
class RateCharge(BaseModel):
    """Individual charge component (ocean freight, surcharge, etc.)"""
    
    charge_type: Literal[
        "ocean_freight", "dg3_surcharge", "dg2_surcharge", 
        "baf_lss", "ams"
    ]
    container_type: Literal["20'GP", "40'GP", "40'HC"]
    amount: Optional[float] = Field(
        None, 
        description="Null if value is 'INCL.' or '-' in PDF"
    )
    currency: Literal["USD", "SGD"]
    is_included: bool = Field(
        default=False,
        description="True only if PDF shows 'INCL.'"
    )
    unit: Literal["per_container", "per_set", "per_teu"] = "per_container"
    
    class Config:
        extra = "forbid"

In [ ]:
class VendorRate(BaseModel):
    """Complete freight rate for a specific port/route"""
    
    route_type: Literal["outbound", "inbound"] = Field(
        description="outbound=POL Singapore, inbound=POD Singapore"
    )
    country: str = Field(
        description="Country name (e.g., BRUNEI, PHILIPPINES, INDIA)"
    )
    port_name: str = Field(
        description="Port name, may include notes like 'MUARA (SGD)' or 'CHATTOGRAM (CY/CY)'"
    )
    region: Literal["sea_china", "india_middle_east_subcon"] = Field(
        description="Region determines effective date range"
    )
    pol: str = Field(description="Port of Loading")
    pod: str = Field(description="Port of Destination")
    charges: List[RateCharge] = Field(
        description="All charges for this route (ocean freight, surcharges, BAF, AMS)"
    )
    effective_date_start: str = Field(
        description="ISO date. '2025-11-01' for sea_china, '2025-11-14' for india_middle_east_subcon"
    )
    effective_date_end: str = Field(
        description="ISO date. '2025-11-30' for both regions"
    )
    special_notes: List[str] = Field(
        default_factory=list,
        description="Any special conditions for this route"
    )
    
    class Config:
        extra = "forbid"

In [ ]:
class GlobalFee(BaseModel):
    """Document-wide fees (BL Fee, Seal Fee, etc.)"""
    fee_name: str
    amount: float
    currency: Literal["USD", "SGD"]
    unit: str = Field(description="per_set, per_container, etc.")
    applies_to: Optional[str] = Field(
        None,
        description="If fee applies to specific routes/countries only"
    )
    
    class Config:
        extra = "forbid"

class CompleteTariffDocument(BaseModel):
    """Complete tariff document extraction result"""
    
    document_name: str = "November 2025 Tariff"
    total_routes: int = Field(
        description="Total number of port routes extracted"
    )
    rates: List[VendorRate] = Field(
        description="All freight rates with regional dates"
    )
    global_fees: List[GlobalFee] = Field(
        default_factory=list,
        description="Document-wide fees like BL Fee SGD200/SET"
    )
    notes: List[str] = Field(
        default_factory=list,
        description="Any additional notes from document footer"
    )
    
    class Config:
        extra = "forbid"

## 3. PDF Preprocessing

Convert PDF to text for extraction.

In [ ]:
def read_pdf_as_text(pdf_path: str) -> str:
    """Read PDF and return as text"""
    with open(pdf_path, 'rb') as f:
        content = f.read()
    
    # For now, we'll read the text-converted PDF
    # In production, use pdfplumber or pdf2image
    return content.decode('utf-8', errors='ignore')

def encode_pdf_as_base64(pdf_path: str) -> str:
    """Encode PDF as base64 for vision API"""
    with open(pdf_path, 'rb') as f:
        return base64.b64encode(f.read()).decode('utf-8')

In [ ]:
# Read the tariff PDF
pdf_path = "Nov Tariff.pdf"
pdf_text = read_pdf_as_text(pdf_path)

print(f"PDF length: {len(pdf_text)} characters")
print(f"\nFirst 500 characters:")
print(pdf_text[:500])

## 4. Structured Extraction with Regional Date Logic

The key to deterministic extraction is the **system prompt** that encodes business rules.

In [ ]:
EXTRACTION_SYSTEM_PROMPT = """You are a freight rate extraction specialist. Extract FCL rates with REGIONAL effective dates.

STEP 1 - IDENTIFY REGIONAL DATE RANGES:
From document footer, extract two date ranges:
- S.E.A/CHINA RATES: Effective 1st November '25 to 31st November '25
- INDIA/MIDDLE EAST/SUBCON RATES: Effective 14th November '25 to 31st November '25

STEP 2 - COUNTRY-REGION MAPPING:

sea_china Region (effective 2025-11-01 to 2025-11-30):
├─ BRUNEI (all ports)
├─ PHILIPPINES (Manila, Cebu, Davao, etc.)
├─ THAILAND (Bangkok, Laem Chabang, Lat Krabang, etc.)
├─ VIETNAM (Ho Chi Minh, etc.)
├─ INDONESIA (Jakarta, etc.)
├─ MALAYSIA (Port Klang variants, etc.)
└─ CHINA (Shanghai, etc.)

india_middle_east_subcon Region (effective 2025-11-14 to 2025-11-30):
├─ MIDDLE EAST (Jebel Ali, BND via JEA, etc.)
├─ INDIA (Nhava Sheva, Mundra, etc.)
├─ PAKISTAN (Karachi, etc.)
└─ BANGLADESH (Chattogram, etc.)

STEP 3 - TABLE STRUCTURE PARSING:

Columns (OUTBOUND/INBOUND tables):
1. Country (may be grouped/repeated)
2. Port Name
3. Ocean Freight 20'GP (USD or SGD if marked)
4. Ocean Freight 40'HC (USD or SGD if marked)
5. DG3 Surcharge (USD/TEU)
6. DG2 Surcharge (USD/TEU)
7. BAF/LSS 20'GP (SGD)
8. BAF/LSS 40'GP/HC (SGD)
9. AMS (USD) - usually per set

STEP 4 - VALUE INTERPRETATION:
- Numeric (e.g., "100.00") → amount=100.00, is_included=false
- "INCL." → amount=null, is_included=true
- "-" or blank → amount=null, is_included=false
- "1200/40GP" → amount=1200.00, extract container type

STEP 5 - CURRENCY RULES:
- Ocean Freight: Check port name for (USD) or (SGD), default USD
- DG Surcharges: Always USD, unit=per_teu
- BAF/LSS: Always SGD, unit=per_container
- AMS: Check header, usually USD, unit=per_set

STEP 6 - POL/POD ASSIGNMENT:
- OUTBOUND section: pol="SINGAPORE", pod=<port_name>
- INBOUND section: pol=<port_name>, pod="SINGAPORE"

STEP 7 - REGIONAL DATE ASSIGNMENT:
For each rate:
1. Identify the country
2. Map country to region using mapping above
3. Assign effective_date_start and effective_date_end based on region
4. Include region field for traceability

CRITICAL RULES:
- Extract EVERY row in the tables
- Create separate VendorRate for each port
- Include ALL charges for each container type
- Assign correct regional dates - this is CRITICAL for business operations
- Do NOT hallucinate ports or data
- If uncertain about a value, use null
"""

In [ ]:
class RateCharge(BaseModel):
    """Individual charge component"""
    
    charge_type: Literal[
        "ocean_freight", "dg3_surcharge", "dg2_surcharge", 
        "baf_lss", "ams"
    ]
    container_type: Literal["20'GP", "40'GP", "40'HC"]
    amount: Optional[float] = Field(
        None, 
        description="Null if 'INCL.' or '-'"
    )
    currency: Literal["USD", "SGD"]
    is_included: bool = Field(
        default=False,
        description="True if 'INCL.' in PDF"
    )
    unit: Literal["per_container", "per_set", "per_teu"] = "per_container"
    
    class Config:
        extra = "forbid"

class VendorRate(BaseModel):
    """Complete freight rate for a port route"""
    
    route_type: Literal["outbound", "inbound"]
    country: str = Field(description="Country name")
    port_name: str = Field(description="Port name with any annotations")
    region: Literal["sea_china", "india_middle_east_subcon"] = Field(
        description="Region determines effective dates"
    )
    pol: str = Field(description="Port of Loading")
    pod: str = Field(description="Port of Destination")
    charges: List[RateCharge] = Field(
        description="All charges for this route"
    )
    effective_date_start: str = Field(
        description="ISO date: '2025-11-01' for sea_china, '2025-11-14' for india_middle_east_subcon"
    )
    effective_date_end: str = Field(
        description="ISO date: '2025-11-30' for all regions"
    )
    special_notes: List[str] = Field(default_factory=list)
    
    class Config:
        extra = "forbid"

class GlobalFee(BaseModel):
    """Document-wide fees"""
    fee_name: str
    amount: float
    currency: Literal["USD", "SGD"]
    unit: str
    applies_to: Optional[str] = None
    
    class Config:
        extra = "forbid"

class CompleteTariffDocument(BaseModel):
    """Complete extraction result"""
    
    document_name: str = "November 2025 Tariff"
    total_routes: int
    rates: List[VendorRate]
    global_fees: List[GlobalFee] = Field(default_factory=list)
    notes: List[str] = Field(default_factory=list)
    
    class Config:
        extra = "forbid"

## 5. Extract Rates Using Structured Outputs

This is the **deterministic extraction** - guaranteed to match schema.

In [ ]:
def extract_freight_rates_structured(pdf_text: str) -> CompleteTariffDocument:
    """Extract rates using OpenAI Structured Outputs (deterministic)"""
    
    client = OpenAI()
    
    response = client.chat.completions.create(
        model="gpt-4o-2024-08-06",  # MUST use this model for structured outputs
        messages=[
            {
                "role": "system",
                "content": EXTRACTION_SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": f"""Extract all FCL freight rates from this tariff document.
                
Return complete VendorRate objects for every port in both OUTBOUND and INBOUND sections.
Pay special attention to regional effective dates.

Document:
{pdf_text}"""
            }
        ],
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "freight_rate_extraction",
                "schema": CompleteTariffDocument.model_json_schema(),
                "strict": True  # Enables constrained decoding
            }
        },
        temperature=0  # Deterministic output
    )
    
    # Parse result - GUARANTEED to match schema
    result = CompleteTariffDocument.model_validate_json(
        response.choices[0].message.content
    )
    
    return result

In [ ]:
# Run extraction
print("Extracting freight rates using OpenAI Structured Outputs...")
print("This may take 5-10 seconds...\n")

tariff_data = extract_freight_rates_structured(pdf_text)

print(f"✓ Extraction complete!")
print(f"✓ Total routes extracted: {tariff_data.total_routes}")
print(f"✓ Global fees: {len(tariff_data.global_fees)}")
print(f"✓ Document notes: {len(tariff_data.notes)}")

## 6. Validate Regional Date Consistency

Verify that rates have correct effective dates based on their region.

In [ ]:
def validate_regional_dates(rate: VendorRate) -> bool:
    """Validate dates match region"""
    
    expected_dates = {
        "sea_china": ("2025-11-01", "2025-11-30"),
        "india_middle_east_subcon": ("2025-11-14", "2025-11-30")
    }
    
    expected_start, expected_end = expected_dates[rate.region]
    
    if rate.effective_date_start != expected_start:
        print(f"❌ {rate.port_name}: Expected start {expected_start}, got {rate.effective_date_start}")
        return False
    
    if rate.effective_date_end != expected_end:
        print(f"❌ {rate.port_name}: Expected end {expected_end}, got {rate.effective_date_end}")
        return False
    
    return True

# Validate all rates
validation_results = [validate_regional_dates(rate) for rate in tariff_data.rates]

if all(validation_results):
    print(f"✓ All {len(tariff_data.rates)} rates have correct regional dates")
else:
    failed = sum(1 for r in validation_results if not r)
    print(f"⚠ {failed} rates have incorrect regional dates")

## 7. Analyze Extraction Results

Group and analyze the extracted rates by region.

In [ ]:
# Group by region
sea_china_rates = [r for r in tariff_data.rates if r.region == "sea_china"]
india_me_rates = [r for r in tariff_data.rates if r.region == "india_middle_east_subcon"]

print("\n=== REGIONAL BREAKDOWN ===")
print(f"\nS.E.A/CHINA Region: {len(sea_china_rates)} routes (Effective Nov 1-30)")
for rate in sea_china_rates[:5]:  # Show first 5
    print(f"  • {rate.country:15} → {rate.port_name:25} ({len(rate.charges)} charges)")
if len(sea_china_rates) > 5:
    print(f"  ... and {len(sea_china_rates) - 5} more")

print(f"\nINDIA/MIDDLE EAST/SUBCON Region: {len(india_me_rates)} routes (Effective Nov 14-30)")
for rate in india_me_rates:
    print(f"  • {rate.country:15} → {rate.port_name:25} ({len(rate.charges)} charges)")

## 8. Inspect Individual Rate Details

Let's examine specific rates to see the extracted structure.

In [ ]:
# Example 1: MANILA (S.E.A region)
manila_rates = [r for r in tariff_data.rates if "MANILA" in r.port_name]

if manila_rates:
    manila = manila_rates[0]
    print("=== MANILA Rate (S.E.A Region) ===")
    print(f"Port: {manila.port_name}")
    print(f"Route: {manila.pol} → {manila.pod}")
    print(f"Region: {manila.region}")
    print(f"Effective: {manila.effective_date_start} to {manila.effective_date_end}")
    print(f"\nCharges:")
    for charge in manila.charges:
        if charge.is_included:
            print(f"  • {charge.charge_type:20} {charge.container_type:8} → INCLUDED")
        else:
            amount_str = f"{charge.amount} {charge.currency}" if charge.amount else "N/A"
            print(f"  • {charge.charge_type:20} {charge.container_type:8} → {amount_str}")

In [ ]:
# Example 2: JEBEL ALI (India/ME region)
jebel_rates = [r for r in tariff_data.rates if "JEBEL" in r.port_name]

if jebel_rates:
    jebel = jebel_rates[0]
    print("\n=== JEBEL ALI Rate (India/ME Region) ===")
    print(f"Port: {jebel.port_name}")
    print(f"Route: {jebel.pol} → {jebel.pod}")
    print(f"Region: {jebel.region}")
    print(f"Effective: {jebel.effective_date_start} to {jebel.effective_date_end}")
    print(f"\nCharges:")
    for charge in jebel.charges:
        if charge.is_included:
            print(f"  • {charge.charge_type:20} {charge.container_type:8} → INCLUDED")
        else:
            amount_str = f"{charge.amount} {charge.currency}" if charge.amount else "N/A"
            print(f"  • {charge.charge_type:20} {charge.container_type:8} → {amount_str}")
    
    print(f"\n⚠ Notice: Jebel Ali starts Nov 14 (not Nov 1)!")

## 9. Business Logic Validation

Apply business rules to validate extracted data.

In [ ]:
def validate_business_rules(rates: List[VendorRate]) -> dict:
    """Validate business rules across all rates"""
    
    results = {
        "total_validated": len(rates),
        "passed": 0,
        "warnings": [],
        "errors": []
    }
    
    for rate in rates:
        passed = True
        
        # Rule 1: Must have at least one charge
        if not rate.charges:
            results["errors"].append(f"{rate.port_name}: No charges found")
            passed = False
            continue
        
        # Rule 2: Ocean freight for 40'HC should be >= 20'GP (typically 2x)
        gp20_freight = next(
            (c.amount for c in rate.charges 
             if c.charge_type == "ocean_freight" 
             and c.container_type == "20'GP" 
             and c.amount is not None), 
            None
        )
        hc40_freight = next(
            (c.amount for c in rate.charges 
             if c.charge_type == "ocean_freight" 
             and c.container_type == "40'HC" 
             and c.amount is not None), 
            None
        )
        
        if gp20_freight and hc40_freight and hc40_freight < gp20_freight:
            results["warnings"].append(
                f"{rate.port_name}: 40'HC ({hc40_freight}) < 20'GP ({gp20_freight}) - unusual"
            )
        
        # Rule 3: DG2 surcharge typically higher than DG3
        dg3 = next(
            (c.amount for c in rate.charges 
             if c.charge_type == "dg3_surcharge" and c.amount), 
            None
        )
        dg2 = next(
            (c.amount for c in rate.charges 
             if c.charge_type == "dg2_surcharge" and c.amount), 
            None
        )
        
        if dg2 and dg3 and dg2 < dg3:
            results["warnings"].append(
                f"{rate.port_name}: DG2 ({dg2}) < DG3 ({dg3}) - verify data"
            )
        
        # Rule 4: Regional dates must be valid
        if not validate_regional_dates(rate):
            results["errors"].append(f"{rate.port_name}: Invalid regional dates")
            passed = False
        
        if passed:
            results["passed"] += 1
    
    return results

# Run validation
validation = validate_business_rules(tariff_data.rates)

print("\n=== VALIDATION RESULTS ===")
print(f"Total rates: {validation['total_validated']}")
print(f"Passed: {validation['passed']}")
print(f"Warnings: {len(validation['warnings'])}")
print(f"Errors: {len(validation['errors'])}")

if validation['warnings']:
    print("\nWarnings:")
    for warning in validation['warnings'][:5]:
        print(f"  ⚠ {warning}")

if validation['errors']:
    print("\nErrors:")
    for error in validation['errors']:
        print(f"  ❌ {error}")

## 10. Query Extracted Data

Demonstrate querying rates based on date validity.

In [ ]:
def find_applicable_rate(
    pol: str, 
    pod: str, 
    shipment_date: date,
    rates: List[VendorRate]
) -> Optional[VendorRate]:
    """Find rate valid for specific shipment date"""
    
    for rate in rates:
        # Match route
        if rate.pol.upper() != pol.upper():
            continue
        if rate.pod.upper() not in pod.upper() and pod.upper() not in rate.pod.upper():
            continue
        
        # Check date validity
        start = date.fromisoformat(rate.effective_date_start)
        end = date.fromisoformat(rate.effective_date_end)
        
        if start <= shipment_date <= end:
            return rate
    
    return None

In [ ]:
# Test Case 1: Manila on Nov 5 (should work - effective from Nov 1)
shipment_date_1 = date(2025, 11, 5)
rate_1 = find_applicable_rate("SINGAPORE", "MANILA", shipment_date_1, tariff_data.rates)

if rate_1:
    print(f"✓ Found rate for Singapore → Manila on {shipment_date_1}")
    print(f"  Region: {rate_1.region}")
    print(f"  Effective: {rate_1.effective_date_start} to {rate_1.effective_date_end}")
    
    # Get 20'GP ocean freight
    ocean_freight = next(
        (c for c in rate_1.charges 
         if c.charge_type == "ocean_freight" and c.container_type == "20'GP"),
        None
    )
    if ocean_freight:
        print(f"  20'GP Rate: {ocean_freight.amount} {ocean_freight.currency}")
else:
    print(f"✗ No rate found for Manila on {shipment_date_1}")

In [ ]:
# Test Case 2: Jebel Ali on Nov 5 (should NOT work - effective from Nov 14)
shipment_date_2 = date(2025, 11, 5)
rate_2 = find_applicable_rate("SINGAPORE", "JEBEL ALI", shipment_date_2, tariff_data.rates)

print(f"\n--- Test: Jebel Ali on {shipment_date_2} ---")
if rate_2:
    print(f"✓ Found rate (unexpected!)")
else:
    print(f"✗ No rate found (CORRECT - rate not effective until Nov 14)")
    print(f"  This demonstrates proper regional date handling!")

In [ ]:
# Test Case 3: Jebel Ali on Nov 20 (should work - after Nov 14)
shipment_date_3 = date(2025, 11, 20)
rate_3 = find_applicable_rate("SINGAPORE", "JEBEL ALI", shipment_date_3, tariff_data.rates)

print(f"\n--- Test: Jebel Ali on {shipment_date_3} ---")
if rate_3:
    print(f"✓ Found rate for Singapore → Jebel Ali")
    print(f"  Region: {rate_3.region}")
    print(f"  Effective: {rate_3.effective_date_start} to {rate_3.effective_date_end}")
    
    ocean_freight = next(
        (c for c in rate_3.charges 
         if c.charge_type == "ocean_freight" and c.container_type == "20'GP"),
        None
    )
    if ocean_freight:
        print(f"  20'GP Rate: {ocean_freight.amount} {ocean_freight.currency}")
else:
    print(f"✗ No rate found (unexpected - should be effective)")

## 11. Export Results

Export the extracted rates as JSON for downstream systems.

In [ ]:
# Export complete document
output = tariff_data.model_dump(mode='json')

with open("extracted_tariff_rates.json", "w") as f:
    json.dump(output, f, indent=2)

print("✓ Exported to: extracted_tariff_rates.json")
print(f"\nSummary:")
print(f"  - {output['total_routes']} vendor rates")
print(f"  - {len(output['global_fees'])} global fees")
print(f"  - {len([r for r in output['rates'] if r['region'] == 'sea_china'])} S.E.A/China routes")
print(f"  - {len([r for r in output['rates'] if r['region'] == 'india_middle_east_subcon'])} India/ME routes")

## 12. Display Sample Rate as JSON

Show the complete structure of one extracted rate.

In [ ]:
# Display first rate as formatted JSON
if tariff_data.rates:
    sample_rate = tariff_data.rates[0]
    print("=== Sample VendorRate Object ===")
    print(json.dumps(sample_rate.model_dump(), indent=2, default=str))

## 13. Global Fees Analysis

In [ ]:
print("\n=== GLOBAL FEES ===")
if tariff_data.global_fees:
    for fee in tariff_data.global_fees:
        applies = f" (applies to: {fee.applies_to})" if fee.applies_to else ""
        print(f"  • {fee.fee_name}: {fee.amount} {fee.currency} {fee.unit}{applies}")
else:
    print("  No global fees extracted")
    print("  Expected: BL Fee SGD200/SET, Seal Fee SGD20/CNTR")

## 14. Regional Statistics

In [ ]:
# Analyze charge patterns by region
def analyze_regional_patterns(rates: List[VendorRate]):
    """Analyze pricing patterns by region"""
    
    patterns = {}
    
    for region in ["sea_china", "india_middle_east_subcon"]:
        region_rates = [r for r in rates if r.region == region]
        
        if not region_rates:
            continue
        
        # Collect all ocean freight amounts for 20'GP
        freight_20gp = []
        for rate in region_rates:
            for charge in rate.charges:
                if (charge.charge_type == "ocean_freight" 
                    and charge.container_type == "20'GP" 
                    and charge.amount is not None):
                    freight_20gp.append(charge.amount)
        
        if freight_20gp:
            patterns[region] = {
                "routes": len(region_rates),
                "min_20gp_freight": min(freight_20gp),
                "max_20gp_freight": max(freight_20gp),
                "avg_20gp_freight": sum(freight_20gp) / len(freight_20gp)
            }
    
    return patterns

patterns = analyze_regional_patterns(tariff_data.rates)

print("\n=== REGIONAL PRICING PATTERNS ===")
for region, stats in patterns.items():
    print(f"\n{region.upper().replace('_', ' ')}:")
    print(f"  Routes: {stats['routes']}")
    print(f"  20'GP Ocean Freight Range: ${stats['min_20gp_freight']:.2f} - ${stats['max_20gp_freight']:.2f} USD")
    print(f"  Average: ${stats['avg_20gp_freight']:.2f} USD")

## 15. Summary

### Key Takeaways:

1. **Deterministic Extraction:** Structured outputs guarantee schema compliance
2. **Regional Date Awareness:** Different effective dates per country/region
3. **Type Safety:** Pydantic validates all data automatically
4. **Zero Parsing Code:** No regex, string splitting, or error handling needed
5. **Production Ready:** Output can go directly to database or API

### Regional Date Rules:
- **S.E.A/CHINA:** Nov 1-30 (Brunei, Philippines, Thailand, Vietnam, Indonesia, Malaysia, China)
- **INDIA/MIDDLE EAST/SUBCON:** Nov 14-30 (Middle East, India, Pakistan, Bangladesh)

### Cost per Extraction:
- Model: gpt-4o-2024-08-06
- Input: ~2,000 tokens
- Output: ~3,000 tokens (15-20 routes)
- **Total: ~$0.03-0.04 per document**